# System-1 Router Baseline — Qwen3.5-0.8B + LoRA (`lets_route` / xRouteBench)

Turn the general-purpose Qwen3.5-0.8B (local base: `models/qwen3.5-0.8b`) into
a **System-1 decision router**: given a query and the candidate-model option pool,
pick — in one forward pass — which model should answer it.

Ground-truth labels come from the counterfactual oracle (`router/prepare_dataset.py`):
- `oracle_target` — the cheapest acceptable model for the query
- `acceptable_models` — every model within ε of the best score

Inputs: `data/processed/routing_{train,valid,test}.jsonl` (stratified).
Runs on the **route_f** kernel (Linux CUDA / macOS MPS).

## 0. Check the machine

In [1]:
import sys, time, json, math, random
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm
from transformers import AutoTokenizer, Qwen3_5ForConditionalGeneration
from peft import LoraConfig, get_peft_model

DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("device:", DEVICE)
if DEVICE == "cuda":
    print("gpu:", torch.cuda.get_device_name(0))

/home/mr/Desktop/Fireworks/.route_f/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: cuda
gpu: NVIDIA GeForce RTX 3060 Laptop GPU


## 1. Data — prepared splits

`router/prepare_dataset.py` already produced stratified train / valid / test:
- `data/processed/routing_train.jsonl`
- `data/processed/routing_valid.jsonl`
- `data/processed/routing_test.jsonl` (held-out)

In [2]:
import sys
from pathlib import Path

# Anchor to the repo root (the notebook may run from experiments/ or the root).
ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw").exists()),
    Path.cwd(),
)
PROC = ROOT / "data" / "processed"

def load_jsonl(path):
    with open(path) as fh:
        return [json.loads(line) for line in fh if line.strip()]

train = load_jsonl(PROC / "routing_train.jsonl")
valid = load_jsonl(PROC / "routing_valid.jsonl")
test  = load_jsonl(PROC / "routing_test.jsonl")
print(f"train={len(train)} valid={len(valid)} test={len(test)}")
print("sample keys:", sorted(train[0].keys()))

train=3939 valid=549 test=3729
sample keys: ['accept_epsilon', 'acceptable_models', 'best_perf', 'candidates', 'cheapest_cost_usd', 'cost_gap_usd', 'cost_tier', 'difficulty', 'embedding_id', 'ground_truth', 'metric', 'oracle_target', 'query', 'spread', 'target_cost_usd', 'task']


## 2. The task & option pool

Each record has a `query`, a list of `candidates` (model id + perf + cost),
and an oracle `oracle_target`. We treat every candidate model as an *option*:
the router must choose the option to route to.

In [3]:
def candidates_from(rec):
    return [c["model"] for c in rec["candidates"]]

def build_prompt(rec):
    lines = [rec["query"].strip()]
    lines.append("\nCandidate models:")
    for i, c in enumerate(rec["candidates"], start=1):
        lines.append(f"{i}. {c['model']}")
    lines.append("\nWhich model id should answer this query?")
    return "\n".join(lines)

r0 = train[0]
print(build_prompt(r0))

[System Instruction]
You are an expert Python programmer. Implement the function with no irrelevant words or comments. Put your code in this format: [BEGIN] <Your Code> [Done]

[User Query]
Task: Write a function to caluclate perimeter of a parallelogram.

Your code should pass these tests:

assert parallelogram_perimeter(10,20)==400
assert parallelogram_perimeter(15,20)==600
assert parallelogram_perimeter(8,9)==144

Candidate models:
1. gpt-oss-20b
2. gemma-2-9b-it
3. llama-3-8b-instruct-lite
4. mistral-small-3-24b-instruct
5. rnj-1-instruct
6. qwen2.5-7b-instruct
7. gpt-oss-120b
8. mistral-7b-instruct-v0.3
9. qwen2.5-7b-instruct-turbo
10. qwen3-coder-next
11. deepseek-v3.1
12. llama-3.3-70b-instruct-turbo
13. llama3-70b-instruct
14. mixtral-8x7b-instruct-v0.1
15. mixtral-8x22b-instruct-v0.1
16. cogito-v2-1-671b
17. llama-4-maverick
18. qwen3-next-80b-a3b-instruct

Which model id should answer this query?


## 3. Zero-shot baseline (before tuning)

Prompt the raw Qwen3.5-0.8B to output a model id and parse it. This measures
the ceiling of *not* tuning — the gap we expect LoRA to close.

In [4]:
MODEL = ROOT / "models" / "qwen3.5-0.8b"
tok = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
model = Qwen3_5ForConditionalGeneration.from_pretrained(
    MODEL, dtype=torch.bfloat16 if DEVICE == "cuda" else torch.float32, trust_remote_code=True
).to(DEVICE)

def ask(texts, max_new=24):
    tok.padding_side = "left"
    batch = tok(texts, return_tensors="pt", padding=True, truncation=True, max_length=512).to(DEVICE)
    with torch.no_grad():
        out = model.generate(**batch, max_new_tokens=max_new, do_sample=False)
    return [tok.decode(o[batch.input_ids.shape[1]:], skip_special_tokens=True).strip() for o in out]

def parse_model_id(text, pool):
    text = text.strip().lower()
    for m in pool:
        if m.lower() in text:
            return m
    return None

def evaluate_zero_shot(sample):
    pool = candidates_from(sample[0])
    right = hit = 0; invalid = 0
    for rec in tqdm(sample):
        pred = parse_model_id(ask([build_prompt(rec)])[0], candidates_from(rec))
        if pred is None:
            invalid += 1
        else:
            right += pred == rec["oracle_target"]
            hit += pred in rec["acceptable_models"]
    n = len(sample)
    return right/n, hit/n, invalid

small = test[:32]
base_oracle, base_accept, base_invalid = evaluate_zero_shot(small)
base_oracle, base_accept, base_invalid

  0%|          | 0/32 [00:00<?, ?it/s][transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
100%|██████████| 32/32 [00:33<00:00,  1.

(0.0, 0.0, 32)

## 4. Make it System 1 — LoRA target

We keep Qwen frozen and add LoRA to the attention (and optionally MLP)
projections. The model still predicts the chosen model id as *one token-or-words*
completion; we supervise only that completion.

In [5]:
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.enable_input_require_grads()

LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj",
                "gate_proj", "up_proj", "down_proj"]
model = get_peft_model(model, LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
    task_type="CAUSAL_LM", target_modules=LORA_TARGETS,
))
model.print_trainable_parameters()

trainable params: 6,389,760 || all params: 859,375,680 || trainable%: 0.7435


In [6]:
def encode(rec, max_len=512):
    prompt = build_prompt(rec)
    answer = rec["oracle_target"]
    full = prompt + "\n" + answer
    prompt_ids = tok(prompt, add_special_tokens=True, truncation=True, max_length=max_len)["input_ids"]
    full_ids = tok(full, add_special_tokens=True, truncation=True, max_length=max_len)["input_ids"]
    if tok.eos_token_id is not None and full_ids[-1] != tok.eos_token_id:
        full_ids.append(tok.eos_token_id)
    labels = list(full_ids)
    for i in range(min(len(prompt_ids), len(labels))):
        labels[i] = -100
    return {"input_ids": full_ids, "labels": labels}

def collate(batch, pad_id):
    seqs = [b["input_ids"] for b in batch]
    maxlen = max(map(len, seqs))
    ids = torch.full((len(seqs), maxlen), pad_id, dtype=torch.long)
    lbl = torch.full((len(seqs), maxlen), -100, dtype=torch.long)
    for i, s in enumerate(seqs):
        ids[i, :len(s)] = torch.tensor(s)
        lbl[i, :len(s)] = torch.tensor(batch[i]["labels"])
    return {"input_ids": ids.to(DEVICE), "labels": lbl.to(DEVICE),
            "attention_mask": ids.ne(pad_id).to(DEVICE)}

## 5. Train — LoRA (two-LR style)

Optimizer with a `2e-4` LR for LoRA weights, weight decay, a cosine warm-up
schedule, gradient clipping. (With no separate head in this baseline, one LR
group is enough; the two-group split becomes relevant once a head is added.)

In [7]:
STEPS = 40          # bump to 400-800 for a real run
BATCH = 2
MAXLEN = 512
PAD = tok.pad_token_id or tok.eos_token_id

opt = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad], lr=2e-4, weight_decay=0.01
)
sched = torch.optim.lr_scheduler.LambdaLR(
    opt, lambda s: min(1, (s + 1) / 20) * 0.5 * (1 + math.cos(math.pi * s / STEPS))
)

tokens = random.Random(7).sample(train, min(len(train), STEPS * BATCH))
model.train()
losses = []
t0 = time.time()
for step in (bar := tqdm(range(STEPS))):
    part = tokens[step * BATCH:(step + 1) * BATCH]
    batch = collate([encode(r, MAXLEN) for r in part], PAD)
    out = model(**batch)
    loss = out.loss
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step(); sched.step(); opt.zero_grad(set_to_none=True)
    losses.append(loss.item())
    bar.set_postfix(loss=f"{sum(losses[-10:]) / max(1, len(losses[-10:])):.3f}")

print(f"train {STEPS} steps in {(time.time() - t0)/60:.1f} min, last loss {losses[-1]:.3f}")

  2%|▎         | 1/40 [00:01<01:09,  1.78s/it, loss=1.495]


OutOfMemoryError: CUDA out of memory. Tried to allocate 972.00 MiB. GPU 0 has a total capacity of 5.67 GiB of which 860.94 MiB is free. Process 931830 has 112.00 MiB memory in use. Including non-PyTorch memory, this process has 4.69 GiB memory in use. Of the allocated memory 4.11 GiB is allocated by PyTorch, and 438.96 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

## 6. Before vs after

In [ ]:
@torch.no_grad()
def evaluate(sample):
    model.eval()
    right = hit = 0
    for rec in tqdm(sample):
        batch = collate([encode(rec, 512)], PAD)
        logits = model(**batch).logits[0, -1]
        pred_tok = logits.argmax().item()
        pred = tok.decode([pred_tok]).strip()
        # exact match against oracle target
        if pred == rec["oracle_target"]:
            right += 1; hit += 1
        else:
            # accept ~ a target model id appears in the generated first token
            pred = parse_model_id(pred, candidates_from(rec))
            if pred:
                right += pred == rec["oracle_target"]
                hit += pred in rec["acceptable_models"]
    n = len(sample)
    return right/n, hit/n

tuned_oracle, tuned_accept = evaluate(small)
print(f"zero-shot: oracle={base_oracle:.3f} acceptable={base_accept:.3f} invalid={base_invalid}")
print(f"tuned   : oracle={tuned_oracle:.3f} acceptable={tuned_accept:.3f}")

## 7. Save the adapter

Save the LoRA adapter; at inference the base weights stay frozen and the
adapter can be merged or served via SGLang.

In [ ]:
OUT_DIR = ROOT / "models" / "router-lora"
model.save_pretrained(OUT_DIR)
tok.save_pretrained(OUT_DIR)
print("saved:", OUT_DIR)